# Sentiment Analysis
Analysing sentiment of movie reviews using Naive Bayes Classification from scratch

In [178]:
# Standard libraries 
from collections import Counter

# Import necessary libraries
import numpy as np
import pandas as pd

# NLP libraries 
import nltk 

## Read the data into a dataframe

In [202]:
train_data = pd.read_csv('sentiment_data.csv', header=0)
train_data.shape

(5, 2)

In [203]:
# Look at the data 
train_data

,review,sentiment
0,just plain boring,negative
1,entirely predictable and lacks energy,negative
2,no surprises and very few laughs,negative
3,very powerful,positive
4,the most fun film of the summer,positive


## Data Preprocessing

In [204]:
# Convert the target variable to 1 - positive and 0 negative 
train_data['sentiment'] = train_data['sentiment'].map( {'negative':0, 'positive':1} )
train_data

,review,sentiment
0,just plain boring,0
1,entirely predictable and lacks energy,0
2,no surprises and very few laughs,0
3,very powerful,1
4,the most fun film of the summer,1


## Naive Bayes Algorithm

### Calculating Document Prior Probabilities

$$\hat{P}(c) = \frac{N_{c}}{N{doc}}$$ 

where $\hat{P}(c)$ is the prior probability, $N_{c}$ the number of documents in our training data with class c and $N_{doc}$ be the total number of documents.

In [205]:
# Calculate document frequencies
total_documents = train_data.shape[0]
positive_docs = train_data[train_data['sentiment'] == 1].shape[0]
negative_docs = train_data[train_data['sentiment'] == 0].shape[0]

# Prior probs
prior_pos = positive_docs/total_documents
prior_neg = negative_docs/total_documents
print(f"Positive prior: {prior_pos} and Negative prior: {prior_neg}")

Positive prior: 0.4 and Negative prior: 0.6


### Create the bag of words 

We'll create a bag of words for positive and negative words as well as the total words. The bag of word contains word counts for each word. 


In [206]:
# Concatenate entries from the data
total_text = ''
positive_text = ''
negative_text = ''

for i in range(train_data.shape[0]):
    row = train_data.iloc[i]
    text = row['review']
    
    # Add to the total text 
    total_text = total_text + text + ' '
    
    # Check pos or negative
    if row['sentiment'] == 1:
        positive_text = positive_text + text + ' '
    else: 
        negative_text = negative_text + text + ' '

In [209]:
# Let's have a look at the texts
print(total_text)
print("\n")
print(positive_text)
print("\n")
print(negative_text)

just plain boring entirely predictable and lacks energy no surprises and very few laughs very powerful the most fun film of the summer 


very powerful the most fun film of the summer 


just plain boring entirely predictable and lacks energy no surprises and very few laughs 


In [210]:
# Create bag of words with word frequency counts
total_bog = Counter(total_text.split())
positive_bog = Counter(positive_text.split())
negative_bog = Counter(negative_text.split())

# Word counts : Collect total number of words in each group
total_words = len(total_bog)
positive_words = len(positive_text.split())
negative_words = len(negative_text.split())
print(total_words, positive_words, negative_words)

20 9 14


In [211]:
total_bog, positive_bog, negative_bog

(Counter({'just': 1,
          'plain': 1,
          'boring': 1,
          'entirely': 1,
          'predictable': 1,
          'and': 2,
          'lacks': 1,
          'energy': 1,
          'no': 1,
          'surprises': 1,
          'very': 2,
          'few': 1,
          'laughs': 1,
          'powerful': 1,
          'the': 2,
          'most': 1,
          'fun': 1,
          'film': 1,
          'of': 1,
          'summer': 1}),
 Counter({'very': 1,
          'powerful': 1,
          'the': 2,
          'most': 1,
          'fun': 1,
          'film': 1,
          'of': 1,
          'summer': 1}),
 Counter({'just': 1,
          'plain': 1,
          'boring': 1,
          'entirely': 1,
          'predictable': 1,
          'and': 2,
          'lacks': 1,
          'energy': 1,
          'no': 1,
          'surprises': 1,
          'very': 1,
          'few': 1,
          'laughs': 1}))

### Calculate maximum likelihood estimate

$$\hat{P}(w_{i}|c) = \frac{count(w_{i}, c)}{\sum_{w \subseteq V}^{} count(w, c)} $$

where $V$ is the vocabulary and we compute fraction of times the word $w_{i}$ appears among all words in all documents of topic $c$. We will also apply Laplacian smoothing to the likelihood estimates.

$$\hat{P}(w_{i}|c) = \frac{count(w_{i}, c) + 1}{(\sum_{w \subseteq V}^{} count(w, c) ) + |V| } $$

In [212]:
# We'll create a dictionary directly for the test data
test_example = "predictable with no fun"

In [215]:
# Create a function to calculate the probabilities and resturn a class
def test_nb(text):
    
    # Create negative and pos probabilities 
    negative_posterior = prior_neg
    positive_posterior = prior_pos
    
    # Iterate through the test set 
    tokens = text.split()
    for token in tokens:
        
        # Check if the word is in our training set 
        if not total_bog.get(token):
            continue
        
        # Get the word frequencies
        pos_freq = positive_bog.get(token, 0)
        neg_freq = negative_bog.get(token, 0)
        
        # Calculate the probabilies 
        pos_prob = (pos_freq + 1) / (positive_words + total_words)
        neg_prob = (neg_freq + 1) / (negative_words + total_words)
        
        # Multiply to the posterior probabilities 
        negative_posterior = negative_posterior * neg_prob
        positive_posterior = positive_posterior * pos_prob
    
    print("\nPositive Probability: ", positive_posterior)
    print("\nNegative Probability: ", negative_posterior)
    print("\nHence, the predicted class is: ", end="")
    
    # Return Classes
    if negative_posterior > positive_posterior:
        print("negative")
    else: 
        print("positive")


In [216]:
# Test the function with our example
test_nb(test_example)


Positive Probability:  3.2801672885317154e-05

Negative Probability:  6.106248727864848e-05

Hence, the predicted class is: negative
